# EDA: Game of 24 Puzzle Distribution

Establishes the empirical baseline before any model runs.
Questions answered here:
1. What fraction of random 4-number puzzles (1–13) are solvable?
2. Does solvability vary by number range (low / mid / high)?
3. What does the difficulty distribution look like — are some puzzles harder than others?
4. Verifier smoke test: does the brute-force solver agree with the verifier on 100% of cases?

In [ ]:
import sys
sys.path.insert(0, '..')

import random
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

from src.data.puzzles import generate_puzzles, PuzzleDataset
from src.verifier.core import verify_solution, brute_force_check

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Generate Labeled Dataset

In [ ]:
SEED = 42
N = 500

dataset = generate_puzzles(n=N, seed=SEED)

print(f"Generated : {len(dataset)} puzzles")
print(f"Solvable  : {len(dataset.solvable)} ({dataset.solve_rate:.1%})")
print(f"Unsolvable: {len(dataset.unsolvable)} ({1-dataset.solve_rate:.1%})")

## 2. Solvability by Max Number (Tier)

In [ ]:
def tier(puzzle):
    top = max(puzzle.numbers)
    if top <= 4: return 'low (1–4)'
    if top <= 9: return 'mid (5–9)'
    return 'high (10–13)'

rows = []
for p in dataset:
    rows.append({'tier': tier(p), 'solvable': p.solvable})

df = pd.DataFrame(rows)
tier_stats = df.groupby('tier')['solvable'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'solved', 'count': 'total', 'mean': 'solve_rate'}
)
tier_stats['solve_rate'] = tier_stats['solve_rate'].map('{:.1%}'.format)
print(tier_stats.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
tier_order = ['low (1–4)', 'mid (5–9)', 'high (10–13)']
rates = [
    df[df['tier'] == t]['solvable'].mean()
    for t in tier_order
]
counts = [df[df['tier'] == t].shape[0] for t in tier_order]

bars = ax.bar(tier_order, rates, color=['#4e79a7', '#f28e2b', '#e15759'], width=0.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylim(0, 1.05)
ax.set_ylabel('Solve Rate')
ax.set_title('Game of 24 Solve Rate by Number Tier')

for bar, rate, count in zip(bars, rates, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{rate:.0%}\n(n={count})', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../docs/api/fig_solve_rate_by_tier.png', bbox_inches='tight')
plt.show()

## 3. Distribution of Max Number Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

max_vals_solved   = [max(p.numbers) for p in dataset.solvable]
max_vals_unsolved = [max(p.numbers) for p in dataset.unsolvable]

bins = range(1, 15)
axes[0].hist(max_vals_solved,   bins=bins, align='left', color='#4e79a7', rwidth=0.8)
axes[0].set_title('Solvable — Max Number Distribution')
axes[0].set_xlabel('Max number in puzzle')
axes[0].set_ylabel('Count')

axes[1].hist(max_vals_unsolved, bins=bins, align='left', color='#e15759', rwidth=0.8)
axes[1].set_title('Unsolvable — Max Number Distribution')
axes[1].set_xlabel('Max number in puzzle')

plt.tight_layout()
plt.savefig('../docs/api/fig_max_number_distribution.png', bbox_inches='tight')
plt.show()

## 4. Verifier Smoke Test

Every puzzle where brute force finds a solution must be accepted by the verifier.
Every puzzle where brute force finds no solution must be rejected. Any failure here
means the verifier has a bug and the RL reward signal is corrupted.

In [ ]:
mismatches = []
for puzzle in dataset:
    expr = brute_force_check(puzzle.numbers_list)
    if expr is not None:
        signal = verify_solution(expr, puzzle.numbers_list)
        if not signal.solved:
            mismatches.append((puzzle.numbers, expr, signal))

if mismatches:
    print(f"FAIL: {len(mismatches)} verifier mismatches")
    for nums, expr, sig in mismatches[:5]:
        print(f"  {nums}  expr={expr}  error={sig.error}")
else:
    print(f"PASS: Verifier agrees with brute force on all {len(dataset)} puzzles.")

## 5. Few-Shot Seed Example Quality Check

In [ ]:
from src.llm.few_shot import SEED_EXAMPLES
from src.verifier.core import extract_expression

print("Verifying seed few-shot examples against the verifier:\n")
all_pass = True
for ex in SEED_EXAMPLES:
    expr = extract_expression(ex['solution'])
    signal = verify_solution(expr or '', list(ex['numbers']))
    status = '✓ PASS' if signal.solved else '✗ FAIL'
    print(f"  {status}  {ex['numbers']}  →  {expr}")
    if not signal.solved:
        print(f"         error: {signal.error}")
        all_pass = False

print()
print('All seed examples valid:', all_pass)

## 6. Summary Table (Sprint 1 Baseline Inputs)

Copy these numbers into `docs/sprints/sprint-01-baseline.md`.

In [ ]:
train, val = dataset.split(train_frac=0.8, seed=SEED)

summary = {
    'Total puzzles generated'   : len(dataset),
    'Solvable (brute force)'    : f"{len(dataset.solvable)} ({dataset.solve_rate:.1%})",
    'Unsolvable'                : f"{len(dataset.unsolvable)} ({1-dataset.solve_rate:.1%})",
    'Train split'               : len(train),
    'Val split'                 : len(val),
    'Verifier smoke test'       : 'PASS' if not mismatches else f'FAIL ({len(mismatches)} errors)',
    'Seed few-shot examples'    : len(SEED_EXAMPLES),
}

print('Sprint 1 Baseline Inputs')
print('=' * 45)
for k, v in summary.items():
    print(f'  {k:<30} {v}')